In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [11]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [13]:
!find /kaggle/input -type f | head -20

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [14]:
df = pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

In [15]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [16]:
df.shape

(50000, 2)

In [17]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [22]:
reviews = df["review"].head(100).tolist()

In [24]:
results = classifier(reviews,truncation = True,max_length = 512)

In [25]:
results[0]

{'label': 'NEGATIVE', 'score': 0.5121210813522339}

In [26]:
df_sample = df.head(100).copy()

In [27]:
df_sample["predicted"] = [r["label"] for r in results]
df_sample["confidence"] = [round(r["score"] , 3) for r in results]

In [28]:
df_sample.head(10)

,review,sentiment,predicted,confidence
0,One of the other reviewers has mentioned that ...,positive,NEGATIVE,0.512
1,A wonderful little production. <br /><br />The...,positive,POSITIVE,0.999
2,I thought this was a wonderful way to spend ti...,positive,POSITIVE,0.999
3,Basically there's a family where a little boy ...,negative,NEGATIVE,0.999
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,POSITIVE,0.999
5,"Probably my all-time favorite movie, a story o...",positive,POSITIVE,1.000
6,I sure would like to see a resurrection of a u...,positive,POSITIVE,0.916
7,"This show was an amazing, fresh & innovative i...",negative,NEGATIVE,1.000
8,Encouraged by the positive comments about this...,negative,NEGATIVE,1.000
9,If you like original gut wrenching laughter yo...,positive,POSITIVE,1.000


In [30]:
from sklearn.metrics import classification_report,accuracy_score

df_sample["sentiment_upper"] = df_sample["sentiment"].str.upper()
accuracy = accuracy_score(df_sample["sentiment_upper"] , df_sample["predicted"])

In [31]:
print(f"Accuracy = {accuracy:.2%}")

Accuracy = 89.00%


In [32]:
print(classification_report(df_sample["sentiment_upper"] , df_sample["predicted"]))

              precision    recall  f1-score   support

    NEGATIVE       0.86      0.97      0.91        58
    POSITIVE       0.94      0.79      0.86        42

    accuracy                           0.89       100
   macro avg       0.90      0.88      0.88       100
weighted avg       0.90      0.89      0.89       100



In [36]:
import re

def cleantext(text):
    text = re.sub('<.*?>','',text)
    text = text.strip()
    return text

df['review'] = df['review'].apply(cleantext)

In [38]:
reviews_clean = df["review"].head(100).tolist()
result_clean = classifier(reviews_clean,truncation = True,max_length = 512)

df_sample2 = df.head(100).copy()
df_sample2['predicted_sentiment'] = [r['label'] for r in result_clean]
df_sample2['confidence'] = [round(r['score'], 3) for r in result_clean]

In [39]:
df_sample2['sentiment_upper'] = df_sample2['sentiment'].str.upper()

accuracy2 = accuracy_score(df_sample2['sentiment_upper'], 
                          df_sample2['predicted_sentiment'])
print(f"Accuracy after cleaning: {accuracy2:.2%}")

Accuracy after cleaning: 88.00%


In [41]:
print(classification_report(df_sample2["sentiment_upper"] , df_sample2["predicted_sentiment"]))

              precision    recall  f1-score   support

    NEGATIVE       0.86      0.95      0.90        58
    POSITIVE       0.92      0.79      0.85        42

    accuracy                           0.88       100
   macro avg       0.89      0.87      0.87       100
weighted avg       0.88      0.88      0.88       100



In [42]:
from transformers import pipeline
import joblib

# Save pipeline
classifier.save_pretrained('sentiment_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

AttributeError: 'TextClassificationPipeline' object has no attribute 'modelcard'